# Phase 2: Forecasting Diagnostics

## The Narrative Focus
In our baseline comparisons, a simple Naive persistence model ($y_t = y_{t-1}$) outperformed LightGBM and Prophet on the aggregate holdout set. *This notebook is the forensic post-mortem.* We will use time series analysis to identify the structural limits of aggregate forecasting and prove the existence of 'Feature Degeneracy' in the holdout period.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- HIGH PRIORITY PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data
from src.data.aggregator import aggregate_to_weekly_chain

transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
df = preprocess_data(transactions, products)
df_weekly = aggregate_to_weekly_chain(df)
print(f"Chain-Level Series Length: {len(df_weekly)} Weeks")

## 1. Macro Trend & Volatility Analysis
*"First, we look at the raw chain-level sales. I want to see if the variance expands over time (which would require a Box-Cox transform to enforce stationarity)."*

In [ ]:
df_weekly['Rolling_Mean'] = df_weekly['Sales'].rolling(12).mean()
df_weekly['Rolling_Std'] = df_weekly['Sales'].rolling(12).std()

plt.figure(figsize=(16, 6))
plt.plot(df_weekly['Week'], df_weekly['Sales'], label='Raw Weekly Sales', color='#bdc3c7', alpha=0.6)
plt.plot(df_weekly['Week'], df_weekly['Rolling_Mean'], label='12-Week Rolling Mean', color='#2980b9', linewidth=2)
plt.plot(df_weekly['Week'], df_weekly['Rolling_Std'], label='12-Week Rolling Volatility', color='#e67e22', linewidth=2, linestyle='--')
plt.title('Aggregate Trend and Volatility Stationarity')
plt.xlabel('Week Number')
plt.ylabel('Units Sold')
plt.legend()
plt.show()

**Insight:** Volatility is relatively stable. Log-transforms at the aggregate level are optional, but we maintain them for parity with the store-level methodology.

## 2. Autocorrelation (Mathematical Proof for Lag Engineering)
*"If a recruiter asks *why* I engineered `Lag_1` and `Lag_52` features for LightGBM, this is the proof. Autocorrelation establishes statistical memory."*

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

plot_acf(df_weekly['Sales'], lags=55, ax=axes[0], color='#2c3e50')
axes[0].set_title('ACF: Long-Term Memory and Cyclicality')
axes[0].axvline(52, color='red', linestyle='--', alpha=0.5, label='52-Week Cycle')
axes[0].legend()

plot_pacf(df_weekly['Sales'], lags=55, ax=axes[1], color='#8e44ad')
axes[1].set_title('PACF: Direct Autoregressive Impact')

plt.tight_layout()
plt.show()

## 3. Structural Decomposition
*"What is Prophet actually attempting to learn? We can deconstruct the signal into Trend, Seasonality, and Noise to visualize the ML objective."*

In [ ]:
decomp = seasonal_decompose(df_weekly['Sales'].values, model='additive', period=52)

fig = decomp.plot()
fig.set_size_inches(14, 10)
plt.suptitle('Structural Time Series Decomposition (52-Week Period)', fontsize=16, y=1.02)
plt.show()

## 4. The Diagnostic Climax: Feature Degeneracy
*"Why did LightGBM fail to beat the naive baseline? The core exogenous regressor it relied on (`Promo`) effectively lost all variance in the holdout set."*

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df_weekly['Week'], df_weekly['Promo'], color='#27ae60', linewidth=2)

holdout_start = df_weekly['Week'].iloc[-12]
holdout_end = df_weekly['Week'].iloc[-1]
ax.axvspan(holdout_start, holdout_end, color='#c0392b', alpha=0.2, label='12-Week Out-of-Sample Holdout')

saturation_level = df_weekly['Promo'].iloc[-12:].mean()

ax.annotate(
    f'Holdout Regime:\n{saturation_level:.1%} Promotional Intensity\n(Regressors lose predictive variance)',
    xy=(holdout_start + 6, saturation_level),
    xytext=(holdout_start - 80, saturation_level - 0.2),
    fontsize=12, fontweight='bold', color='#c0392b',
    bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#c0392b', alpha=0.9),
    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2)
)

ax.set_title('Diagnostic Evidence: Promo Regime Saturation', fontsize=16, fontweight='bold')
ax.set_xlabel('Week Number')
ax.set_ylabel('% of Stores on Promotion')
ax.legend(loc='lower left')
plt.show()

## 5. Residual Analysis: Did the Naive Baseline capture everything?
*"If the naive model's residuals look like white noise, it proves that there was no remaining signal left for the ML to capture at the aggregate level."*

In [ ]:
df_weekly['Naive_Forecast'] = df_weekly['Sales'].shift(1)
df_weekly['Naive_Error'] = df_weekly['Sales'] - df_weekly['Naive_Forecast']
holdout_errors = df_weekly.tail(12)['Naive_Error']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(holdout_errors, kde=True, ax=axes[0], color='#7f8c8d')
axes[0].set_title('Holdout Error Distribution (White Noise Check)')
axes[0].set_xlabel('Residual (Actual - Naive)')

# Simple QQ Plot proxy (Sorted errors)
axes[1].plot(np.sort(holdout_errors), np.linspace(0, 1, len(holdout_errors)), marker='o', linestyle='none', color='#2c3e50')
axes[1].set_title('Normal Probability Plot Proxy')
axes[1].set_ylabel('Empirical CDF')
axes[1].set_xlabel('Sorted Residuals')

plt.tight_layout()
plt.show()

***Conclusion:*** *The combination of feature degeneracy (saturated promotions) and a strong autoregressive signature explains the Naive model's dominance. The strategic pivot for business value is clear: stop forecasting the chain, and start forecasting the individual stores where localized variance still exists.*